# 🎙️ TTS Text Optimizer for DesiVocal.com (Enhanced with Chunking)

**Optimize translated text with proper punctuation for natural voice generation.**

This notebook allows you to:
1.  **Setup Ollama**: Install and run Ollama locally in Colab.
2.  **Download Model**: Choose and pull a high-quality LLM (e.g., Qwen2.5, TranslateGemma).
3.  **Optimize Text**: Upload your text file, and the AI will add proper punctuation for natural TTS flow.
4.  **Download Result**: Save the optimized text as a `.txt` file for use on DesiVocal.com.

**✨ NEW: Supports large texts with automatic chunking and progress tracking!**

## 📦 Step 1: Install & Setup Ollama
Run this cell to install Ollama and start the server in the background.

In [ ]:
# Install required packages
!pip install -q ollama requests ipywidgets

# Install and start Ollama server
import subprocess
import time
import os
import sys

print("🦙 Installing Ollama...")

# Install zstd first (required for Ollama extraction)
!apt-get update -qq && apt-get install -y -qq zstd > /dev/null 2>&1

# Download and install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

print("\n🚀 Starting Ollama server in background...")

# Start Ollama server in background
os.environ['OLLAMA_HOST'] = '127.0.0.1:11434'
subprocess.Popen(['/usr/local/bin/ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to start
time.sleep(5)

# Verify server is running
try:
    import ollama
    ollama.list()
    print("✅ Ollama server is running and ready!")
except Exception as e:
    print(f"⚠️ Ollama server may not be ready yet. Error: {e}")
    print("   Please wait a few seconds and try running the next cell.")

🦙 Installing Ollama...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

🚀 Starting Ollama server in background...
✅ Ollama server is running and ready!


## 📥 Step 2: Download Model
Select the model you want to use for optimization. `qwen2.5:14b` is recommended for high quality.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import ollama

print("🦙 Ollama Model Selection")
print("=" * 30)

# Model options
OLLAMA_MODELS = {
    "gemma3:27b (Google's Best Format)": "gemma3:27b",
    "qwen2.5:14b (Recommended - High Quality)": "qwen2.5:14b",
    "gpt-oss:20b (OPENAI's Thinking Format)": "gpt-oss:20b",
    "qwen3:14b (QWEN's low parameter thinking - AI recommended)": "qwen3:14b",
}

model_dropdown = widgets.Dropdown(
    options=list(OLLAMA_MODELS.keys()),
    value="gemma3:27b (Google's Best Format)",
    description='Model:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

display(model_dropdown)
print("\nSelect a model and run this cell to download it.")

🦙 Ollama Model Selection


Dropdown(description='Model:', layout=Layout(width='400px'), options=("gemma3:27b (Google's Best Format)", 'qw…


Select a model and run this cell to download it.


In [ ]:
# Pull the selected model
selected_model_name = OLLAMA_MODELS[model_dropdown.value]
print(f"📥 Pulling model: {selected_model_name}...")
print("   This may take a few minutes.")

try:
    # Pull with stream to show progress (simplified for non-interactive output)
    current_digest = ''
    for progress in ollama.pull(selected_model_name, stream=True):
        digest = progress.get('digest', '')
        if digest != current_digest and current_digest:
             print() # Newline
        current_digest = digest

        status = progress.get('status', '')
        if 'completed' in progress and 'total' in progress:
             completed = progress['completed']
             total = progress['total']
             pct = (completed / total * 100) if total > 0 else 0
             print(f"\r   {status}: {pct:.1f}%", end='', flush=True)
        else:
             print(f"\r   {status}", end='', flush=True)

    print(f"\n\n✅ Model '{selected_model_name}' ready to use!")
except Exception as e:
    print(f"\n❌ Error pulling model: {e}")

📥 Pulling model: gemma3:27b...
   This may take a few minutes.
   pulling e796792eba26: 100.0%
   pulling e0a42594d802: 100.0%
   pulling dd084c7d92a3: 100.0%
   pulling 3116c5225075: 100.0%
   pulling f838f048d368: 100.0%
   success

✅ Model 'gemma3:27b' ready to use!


## 🧠 Step 3: Define Enhanced Optimizer Class with Chunking
This code defines the logic to communicate with Ollama and optimize the text with support for large texts.

In [ ]:
import requests
import json
import sys
import re
import time

class TTSOptimizer:
    """Optimizes text for desivocal.com TTS generation with chunking support"""

    def __init__(self, model_name="translategemma:27b", chunk_size=2000, timeout=6000):
        """
        Initialize the TTS Optimizer

        Args:
            model_name: Name of the Ollama model to use
            chunk_size: Maximum characters per chunk (default: 2000)
            timeout: Request timeout in seconds (default: 600)
        """
        self.ollama_url = "http://localhost:11434/api/generate"
        self.model = model_name
        self.chunk_size = chunk_size
        self.timeout = timeout
        print(f"🤖 Initialized TTS Optimizer")
        print(f"   Model: {self.model}")
        print(f"   Chunk size: {self.chunk_size} chars")
        print(f"   Timeout: {self.timeout}s per chunk")

    def chunk_text(self, text: str) -> list:
        """
        Split text into chunks at sentence boundaries

        Args:
            text: Input text to chunk

        Returns:
            List of text chunks
        """
        if len(text) <= self.chunk_size:
            return [text]

        chunks = []
        current_chunk = ""

        # Split by common sentence endings (periods, question marks, exclamation marks)
        # This regex tries to split on sentence boundaries while preserving the punctuation
        sentences = re.split(r'([।॥.!?।]\s+)', text)

        for i in range(0, len(sentences), 2):
            sentence = sentences[i]
            separator = sentences[i+1] if i+1 < len(sentences) else ""

            # Check if adding this sentence would exceed chunk size
            if len(current_chunk) + len(sentence) + len(separator) > self.chunk_size and current_chunk:
                chunks.append(current_chunk.strip())
                current_chunk = sentence + separator
            else:
                current_chunk += sentence + separator

        # Add the last chunk if not empty
        if current_chunk.strip():
            chunks.append(current_chunk.strip())

        # If we still have no chunks (no sentence boundaries found), split by character count
        if not chunks:
            chunks = [text[i:i+self.chunk_size] for i in range(0, len(text), self.chunk_size)]

        print(f"\n📊 Split text into {len(chunks)} chunks")
        for idx, chunk in enumerate(chunks, 1):
            print(f"   Chunk {idx}: {len(chunk)} characters")

        return chunks

    def get_optimization_prompt(self, text: str, language: str = "Hindi") -> str:
        prompt = f"""You are an expert text formatter for DesiVocal.com TTS system. Format Hindi text for optimal single-voice TTS output with clear speaker identification.

═══════════════════════════════════════════════════════════════
RULE 1: WORD PRESERVATION
═══════════════════════════════════════════════════════════════

CRITICAL: Every input word MUST appear in output.
Exception: Attribution words like "ने कहा", "ने पूछा" become speaker tags.

ALLOWED:
- Modify numbers, dates, symbols, punctuation
- ADD speaker tags (होम्स:, वाटसन:)
- ADD different dialogue punctuation per speaker

FORBIDDEN:
- Adding explanations, metadata, annotations
- Translating between languages
- Dropping words
- Adding your own content

═══════════════════════════════════════════════════════════════
RULE 2: IDENTIFY SPEAKERS FROM CONTEXT
═══════════════════════════════════════════════════════════════

DesiVocal uses ONE VOICE. You must explicitly mark who's speaking.

LOOK FOR CONTEXT CLUES:
"होम्स ने कहा" → Speaker is होम्स
"राज ने पूछा" → Speaker is राज
"मैंने जवाब दिया" → Speaker is narrator (मैं)
"उसने कहा" → Look back 2-3 sentences to identify who "उस" is

PRIORITY ORDER:
1. Named characters (होम्स, वाटसन, राज, सीमा)
2. Roles/titles (राजा, डाक्टर, महाराज, काउंट)
3. Resolved pronouns (track who "उसने" refers to)
4. LAST RESORT: वक्ता1, वक्ता2 (only if no context)

TRACK CONVERSATIONS:
In dialogue between होम्स and वाटसन, speakers alternate:
First quote → होम्स, Second quote → वाटसन, Third quote → होम्स
Continue alternating unless context indicates otherwise.

EXAMPLES:

INPUT:
होम्स ने कहा, "यह जरूरी है।" वाटसन ने पूछा, "क्यों?" "क्योंकि," होम्स ने समझाया।

OUTPUT:
होम्स: 'यह जरूरी है.'
वाटसन: "क्यों?"
होम्स: 'क्योंकि,' [continue]

---

INPUT:
राज ने सीमा से कहा, "चलो।" "ठीक है," उसने कहा। "कब?" राज ने पूछा।

OUTPUT:
राज: «चलो.»
सीमा: *ठीक है.*
राज: «कब?»

---

INPUT:
उसने मुझे देखा और कहा, "क्या तुम आओगे?" मैंने जवाब दिया, "हाँ।"

OUTPUT:
[Identify "उस" from context - assume होम्स]
होम्स: 'क्या तुम आओगे?'
मैं: "हाँ."

═══════════════════════════════════════════════════════════════
RULE 3: DIFFERENT PUNCTUATION PER SPEAKER
═══════════════════════════════════════════════════════════════

CRITICAL: Use DIFFERENT punctuation for EACH character.
This helps listeners distinguish speakers in single-voice TTS.

PUNCTUATION ASSIGNMENT:
Main protagonist → 'single quotes'
Secondary character → "double quotes"
Authority/antagonist → *asterisks*
Additional characters → «guillemets»

EXAMPLE MAPPING:
होम्स → 'single quotes'
वाटसन → "double quotes"
राजा/महाराज/काउंट → *asterisks*
आइरीन एडलर → «guillemets»

CONSISTENCY: Each character keeps same punctuation throughout ENTIRE text.

FORMAT:
SpeakerName: [mark]dialogue[mark]

होम्स: 'यह बहुत जरूरी है.'
वाटसन: "मुझे पता है."
राजा: *मैं चिंतित हूँ.*
आइरीन: «मैं नहीं डरती.»

REMOVE ATTRIBUTION WORDS:
WRONG: होम्स ने कहा: 'मैं जा रहा हूं.'
RIGHT: होम्स: 'मैं जा रहा हूं.'

SEPARATION:
Each speaker on NEW LINE with blank line before.

═══════════════════════════════════════════════════════════════
RULE 4: TECHNICAL FORMATTING
═══════════════════════════════════════════════════════════════

1. ROMAN NUMERALS → REGULAR NUMBERS
I→1, II→2, III→3, IV→4, V→5, VI→6, VII→7, VIII→8, IX→9, X→10
Chapter I → Chapter 1
भाग II → भाग 2
I. → 1.

2. NUMBERS: Remove commas
50,000 → 50000
1,50,000 → 150000

3. DATES: Use month names
15/03/2024 → 15 मार्च 2024

4. YEARS: Add "सन" prefix (no space)
1988 → सन1988
"वह 1995 में पैदा हुआ" → "वह सन1995 में पैदा हुआ"

5. TIME: Write in words
3:30 PM → साढ़ेतीन or तीन बजकर तीस मिनट

6. THE "10" BUG (CRITICAL)
DesiVocal doesn't speak "10" or "दस" properly. ALWAYS use "ten":
10 books → ten किताबें
Chapter 10 → Chapter ten
10:00 → ten बजे
8-10 → 8सेten

7. RANGES: Use "से" (no space)
5-8 → 5से8
10-15 → tenसे15

8. PERCENTAGES
50% → 50 percent

9. ABBREVIATIONS
डॉ. → डाक्टर
रु. → रुपये
Keep: श्री, श्रीमती

10. ACRONYMS: Remove periods
U.S.A. → USA
N.A.S.A. → NASA

11. EMAILS & URLS
@ → at the rate
. → dot (in email/URL context only)
hr@company.com → hr at the rate company dot com

12. COMPOUND WORDS: Remove hyphens
क्रॉस-चेक → क्रॉस चेक

13. SYMBOLS
°F → डिग्री फ़ारेनहाइट
°C → डिग्री सेल्सियस
× → गुना

PUNCTUATION FOR PACING (non-dialogue):
, = short pause
| = medium pause (context shift)
. = long pause
,, = extended pause
... = suspense
!! = excitement
?? = confusion

═══════════════════════════════════════════════════════════════
COMPREHENSIVE EXAMPLES
═══════════════════════════════════════════════════════════════

EXAMPLE 1: Book dialogue with context

INPUT:
अध्याय I
यह 15 मार्च, 1988 की बात है। होम्स ने कहा, "मैं 10 बजे आऊंगा।" वाटसन ने पूछा, "क्यों?" "क्योंकि यह जरूरी है," होम्स ने कहा।

OUTPUT:
अध्याय 1.

यह 15 मार्च, सन1988 की बात है.

होम्स: 'मैं ten बजे आऊंगा.'

वाटसन: "क्यों?"

होम्स: 'क्योंकि यह जरूरी है.'

---

EXAMPLE 2: Alternating dialogue

INPUT:
राज ने कहा, "चलो बाजार चलें।" सीमा ने कहा, "अभी?" "हाँ," राज ने कहा। "ठीक है," सीमा ने कहा।

OUTPUT:
राज: «चलो बाजार चलें.»

सीमा: *अभी?*

राज: «हाँ.»

सीमा: *ठीक है.*

---

EXAMPLE 3: Pronoun resolution

INPUT:
होम्स कमरे में था। वाटसन ने उससे पूछा, "क्या तुम आओगे?" उसने जवाब दिया, "हाँ।" "कब?" वाटसन ने पूछा।

OUTPUT:
होम्स कमरे में था.

वाटसन: "क्या तुम आओगे?"

होम्स: 'हाँ.'

वाटसन: "कब?"

---

EXAMPLE 4: Narrator speaking

INPUT:
मैंने होम्स से कहा, "यह अजीब है।" "बिल्कुल," उसने कहा। मैंने सोचा कि वह सही था।

OUTPUT:
मैं: "यह अजीब है."

होम्स: 'बिल्कुल.'

मैंने सोचा कि वह सही था.

---

EXAMPLE 5: Complex conversation

INPUT:
राजा ने होम्स से कहा, "मुझे मदद चाहिए।" होम्स ने पूछा, "किस बात में?" "एक तस्वीर," राजा ने कहा। वाटसन ने टिप्पणी की, "यह दिलचस्प है।"

OUTPUT:
राजा: *मुझे मदद चाहिए.*

होम्स: 'किस बात में?'

राजा: *एक तस्वीर.*

वाटसन: "यह दिलचस्प है."

═══════════════════════════════════════════════════════════════
PROCESSING WORKFLOW
═══════════════════════════════════════════════════════════════

STEP 1: Read entire text, identify all characters
STEP 2: Assign punctuation:
- Main character → 'single quotes'
- Secondary → "double quotes"
- Authority → *asterisks*
- Others → «guillemets»
STEP 3: Format dialogue with speaker tags
STEP 4: Apply technical formatting
STEP 5: Verify consistency

VERIFICATION CHECKLIST:
All Roman numerals → regular numbers
All "10" → "ten"
All years → "सन" prefix
Numbers → no commas
Speakers → identified from context (minimal वक्ता1, वक्ता2)
Each character → consistent punctuation
Attribution words → removed
Email/URL dots → "dot"
Word count → preserved

═══════════════════════════════════════════════════════════════
OUTPUT INSTRUCTIONS
═══════════════════════════════════════════════════════════════

Return ONLY the formatted text.
NO explanations.
NO bullet points.
NO metadata.
Just clean formatted text ready for TTS.

═══════════════════════════════════════════════════════════════
INPUT TEXT: {text}
"""
        return prompt

    def optimize_chunk(self, chunk: str, language: str = "Hindi", retry_count: int = 3) -> str:
        """
        Optimize a single chunk of text with retry logic

        Args:
            chunk: Text chunk to optimize
            language: Target language
            retry_count: Number of retries on failure

        Returns:
            Optimized text chunk
        """
        prompt = self.get_optimization_prompt(chunk, language)

        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": 0.5,
                "top_p": 0.9,
                "num_predict": -1
            }
        }

        for attempt in range(retry_count):
            try:
                response = requests.post(self.ollama_url, json=payload, timeout=self.timeout)
                response.raise_for_status()
                result = response.json()
                optimized_text = result.get("response", "").strip()
                # Clean output
                optimized_text = self._clean_output(optimized_text)
                return optimized_text
            except requests.exceptions.Timeout:
                if attempt < retry_count - 1:
                    wait_time = (attempt + 1) * 10
                    print(f"\n⚠️ Timeout on attempt {attempt + 1}/{retry_count}. Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n❌ Failed after {retry_count} attempts due to timeout")
                    raise
            except Exception as e:
                if attempt < retry_count - 1:
                    wait_time = (attempt + 1) * 5
                    print(f"\n⚠️ Error on attempt {attempt + 1}/{retry_count}: {e}")
                    print(f"   Retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"\n❌ Failed after {retry_count} attempts: {e}")
                    raise

        return None

    def optimize(self, text: str, language: str = "Hindi") -> str:
        """
        Optimize text with automatic chunking for large texts

        Args:
            text: Input text to optimize
            language: Target language

        Returns:
            Optimized text
        """
        # Split text into chunks
        chunks = self.chunk_text(text)

        if len(chunks) == 1:
            print(f"\n📤 Processing single chunk ({len(text)} chars)...")
            return self.optimize_chunk(chunks[0], language)

        # Process multiple chunks
        print(f"\n🔄 Processing {len(chunks)} chunks...")
        optimized_chunks = []

        for idx, chunk in enumerate(chunks, 1):
            print(f"\n📤 Processing chunk {idx}/{len(chunks)} ({len(chunk)} chars)...")
            try:
                optimized = self.optimize_chunk(chunk, language)
                if optimized:
                    optimized_chunks.append(optimized)
                    print(f"✅ Chunk {idx}/{len(chunks)} complete!")
                else:
                    print(f"❌ Chunk {idx}/{len(chunks)} failed - using original")
                    optimized_chunks.append(chunk)
            except Exception as e:
                print(f"❌ Error processing chunk {idx}: {e}")
                print("   Using original chunk text")
                optimized_chunks.append(chunk)

        # Combine all chunks
        final_text = " ".join(optimized_chunks)
        print(f"\n✅ All chunks processed! Total output: {len(final_text)} characters")
        return final_text

    def _clean_output(self, text: str) -> str:
        """Clean the model output to remove formatting artifacts"""
        text = text.replace("```", "").replace("**", "")
        lines = [line.strip() for line in text.split('\n')
                 if line.strip() and not line.strip().startswith('#') and not line.strip().startswith('OUTPUT')]
        return '\n'.join(lines).strip()

print("✅ TTSOptimizer class loaded with chunking support!")

✅ TTSOptimizer class loaded with chunking support!


## 📝 Step 4: Upload & Optimize
Upload your `.txt` file containing the text to optimize, select the language, and run the optimization.

In [ ]:
from google.colab import files
import ipywidgets as widgets
from IPython.display import display

print("📂 Please upload your text file (.txt):")
uploaded = files.upload()

if uploaded:
    uploaded_filename = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {uploaded_filename}")
else:
    print("⚠️ No file uploaded yet.")

language_input = widgets.Text(
    value='Hindi',
    placeholder='Target Language',
    description='Language:',
    layout=widgets.Layout(width='300px')
)

# Chunk size selector
chunk_size_input = widgets.IntSlider(
    value=2000,
    min=500,
    max=5000,
    step=100,
    description='Chunk Size:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

print("\n⚙️ Configuration:")
display(language_input)
display(chunk_size_input)
print("\nℹ️ Chunk size: smaller = more API calls but less timeout risk")
print("   Recommended: 1500-2500 for large models, 2500-4000 for smaller models")

📂 Please upload your text file (.txt):


Saving translation_hin_20260212_102003.txt to translation_hin_20260212_102003.txt
✅ Uploaded: translation_hin_20260212_102003.txt

⚙️ Configuration:


Text(value='Hindi', description='Language:', layout=Layout(width='300px'), placeholder='Target Language')

IntSlider(value=2000, description='Chunk Size:', layout=Layout(width='400px'), max=5000, min=500, step=100, st…


ℹ️ Chunk size: smaller = more API calls but less timeout risk
   Recommended: 1500-2500 for large models, 2500-4000 for smaller models


In [ ]:
# Run Optimization on Uploaded File
if not uploaded:
    print("⚠️ Please upload a file in the previous step first!")
else:
    # Read file content
    try:
        text_content = uploaded[uploaded_filename].decode("utf-8")
        print(f"📄 Read {len(text_content)} characters from file.")
        print(f"📏 File size: {len(text_content):,} characters")

        # Initialize optimizer with selected model and chunk size
        try:
            model_to_use = selected_model_name
        except NameError:
            model_to_use = "qwen2.5:14b" # Fallback
            print("⚠️ Using default model: qwen2.5:14b")

        optimizer = TTSOptimizer(
            model_name=model_to_use,
            chunk_size=chunk_size_input.value,
            timeout=600  # 10 minutes per chunk
        )

        print(f"\n⏳ Starting optimization for {language_input.value}...")
        print("=" * 50)

        start_time = time.time()
        optimized_text = optimizer.optimize(text_content, language=language_input.value)
        end_time = time.time()

        processing_time = end_time - start_time

        if optimized_text:
            print("\n" + "=" * 50)
            print("✨ OPTIMIZATION COMPLETE!")
            print("=" * 50)
            print(f"⏱️  Processing time: {processing_time:.1f} seconds ({processing_time/60:.1f} minutes)")
            print(f"📊 Input length: {len(text_content):,} chars")
            print(f"📊 Output length: {len(optimized_text):,} chars")
            print(f"📈 Size change: {((len(optimized_text) - len(text_content)) / len(text_content) * 100):+.1f}%")

            print("\n✨ Preview (First 800 characters):")
            print("=" * 50)
            print(optimized_text[:800])
            if len(optimized_text) > 800:
                print("\n... (truncated)")
            print("=" * 50)

            # Save to file
            output_filename = f"optimized_{uploaded_filename}"
            with open(output_filename, 'w', encoding='utf-8') as f:
                f.write(optimized_text)

            print(f"\n💾 Saved to: {output_filename}")
            print("📥 Downloading file...")

            # Trigger download
            files.download(output_filename)
            print("\n✅ Done! Check your downloads folder.")

        else:
            print("\n❌ Optimization failed. Please check the errors above.")

    except Exception as e:
        print(f"\n❌ Error reading or processing file: {e}")
        import traceback
        print("\n📋 Full error details:")
        print(traceback.format_exc())

📄 Read 43484 characters from file.
📏 File size: 43,484 characters
🤖 Initialized TTS Optimizer
   Model: gemma3:27b
   Chunk size: 1500 chars
   Timeout: 600s per chunk

⏳ Starting optimization for Hindi...

📊 Split text into 31 chunks
   Chunk 1: 1370 characters
   Chunk 2: 1474 characters
   Chunk 3: 1449 characters
   Chunk 4: 1465 characters
   Chunk 5: 1496 characters
   Chunk 6: 1359 characters
   Chunk 7: 1459 characters
   Chunk 8: 1444 characters
   Chunk 9: 1419 characters
   Chunk 10: 1483 characters
   Chunk 11: 1493 characters
   Chunk 12: 1436 characters
   Chunk 13: 1484 characters
   Chunk 14: 1432 characters
   Chunk 15: 1496 characters
   Chunk 16: 1492 characters
   Chunk 17: 1355 characters
   Chunk 18: 1439 characters
   Chunk 19: 1391 characters
   Chunk 20: 1447 characters
   Chunk 21: 1434 characters
   Chunk 22: 1475 characters
   Chunk 23: 1441 characters
   Chunk 24: 1486 characters
   Chunk 25: 1485 characters
   Chunk 26: 1464 characters
   Chunk 27: 1429 ch

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Done! Check your downloads folder.


## 🔧 Troubleshooting Tips

**If you're still getting timeouts:**

1. **Reduce chunk size**: Try 1000-1500 characters instead of 2000
2. **Use a smaller model**: Switch to `qwen2.5:7b` or `mistral:7b`
3. **Check Ollama server**: Run `!ollama ps` to see if the model is loaded
4. **Restart Ollama**: Go back to Step 1 and re-run the server setup

**For very large files (100k+ characters):**
- Consider splitting your file manually into smaller files first
- Use chunk size of 1000 characters or less
- The processing will take longer but will be more reliable